# 03a - Build Strategy Datasets

CPU only. Supply your own strategy positions; this stage does not train a model. All eleven themes are supported. PGN comments and annotations are not imported.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## Project and Dependencies

Keep Colab's existing CUDA PyTorch. Restart only if pip explicitly requires it.

In [ ]:
from pathlib import Path
import sys
import subprocess

PROJECT_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/Education/Deep Reinforcement Learning/Chess")
if not (PROJECT_ROOT / "chess_rl").is_dir():
    raise FileNotFoundError(f"Project files are missing from {PROJECT_ROOT}. See README.md.")
sys.path.insert(0, str(PROJECT_ROOT))
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "-r", str(PROJECT_ROOT / "requirements_colab.txt")])

## Run Configuration

Edit configs/default.yaml once for the workflow. Use a new run_id for a changed experiment.

In [ ]:
from chess_rl.config import load_config, prepare_directories
from chess_rl.reproducibility import metadata, read_json, atomic_json, sha256

prepare_directories(PROJECT_ROOT)
cfg = load_config(PROJECT_ROOT, "strategy.yaml")
print("Run:", cfg["run_id"])
print("Runtime:", metadata())

## Configure Sources

Edit configs/strategy.yaml: strategy_dataset.sources accepts PGN, one-FEN-per-line, CSV or JSONL. Set theme/subtheme there or in each CSV/JSONL row. A line is a JSON list of legal UCI moves. Converted Lichess Puzzle DB and STS-Rating files belong here; motif detector tags must be written before this notebook runs. Played moves and lines are observations unless best_move/policy_target is explicitly supplied. Optional UCI labelling uses one CPU thread and fails clearly if its configured executable is missing.

In [ ]:
from chess_rl.strategy_taxonomy import TAXONOMY
for theme, subthemes in TAXONOMY.items():
    print(theme, ":", ", ".join(subthemes))
print("Sources:", cfg["strategy_dataset"]["sources"])

## Build and Persist

Whole source games share a split. Positions duplicated across partitions are removed. Notebook 01 fixes source partitions and reserved openings before either 02 or 03a. Annotated benchmark games default to test. PGNs with common early openings therefore lose those shared positions. Supply enough independent games.

In [ ]:
from chess_rl.strategy_dataset import build_strategy_datasets
manifest = build_strategy_datasets(PROJECT_ROOT, cfg)
print("Split counts:", manifest["counts"])
print("Saved to:", PROJECT_ROOT / "data/strategy" / cfg["strategy_dataset"]["dataset_id"])

## Summary Tables and Charts

CSV and JSONL share the documented schema. Small corpora may have empty partitions; 03b requires labelled train and validation rows. No labels are fabricated to fill a gap.

In [ ]:
from chess_rl.strategy_plots import dataset_summary
display(dataset_summary(PROJECT_ROOT, cfg["run_id"], manifest))

## History and Heuristic Limits

FEN validity checks basic chess constraints, not historical reachability. PGN history is retained as moves for repetition diagnostics. FEN alone cannot reveal repetition or prior castling. Backward pawns, outposts, trapped pieces, bad bishops and fortress-like positions are diagnostics, not adjudications or training labels. Sacrifice, skewer and annotated-game themes require curated tags.